# 24. Real-world CSVs → SQL

*Module: 6 — Python access · Companion summary: `lectures/24-real-world-csv-loading.md` · SQL engine: SQLite (in-memory)*

## What you will study

- read a CSV with pandas and push it into SQL with to_sql
- load data where it isn't in a database yet
- handle identifiers with backticks / double quotes

---
## Setup — build the sample database this notebook runs on
*Run the cell below first: it creates an in-memory SQLite database and the `q()` helper that prints every query's result set.*

## 🧪 Synthesize a small `dogs.csv` (course's dataset flavor: a real-world CSV)

We write one with pandas, then do the full CSV → SQL pipeline. `tempfile` keeps
re-runs clean:

In [ ]:
import os, sqlite3, tempfile
import pandas as pd

csv_path = os.path.join(tempfile.mkdtemp(), 'dogs.csv')
df = pd.DataFrame({
    'name':   ['Max', 'Bella', 'Rocky', 'Luna'],
    'breed':  ['Labrador', 'Poodle', 'Boxer', 'Husky'],
    'age':    [5, 3, 8, 2],
    'weight_kg': [30.0, 6.5, 32.0, 26.0],
})
df.to_csv(csv_path, index=False)
print(open(csv_path).read())

## 🧪 read the CSV straight back with pandas

In [ ]:
dogs = pd.read_csv(csv_path)
dogs

## 🧪 push it into SQLite — read_sql_query to get it back

In [ ]:
conn = sqlite3.connect(':memory:')
dogs.to_sql('dogs', conn, index=False, if_exists='replace')   # the magic line
pd.read_sql_query("SELECT * FROM dogs", conn)

## 🧪 ... or with our q() helper (raw sqlite3, no pandas for the result)

In [ ]:
import sqlite3
import textwrap

conn = sqlite3.connect(':memory:')  # throw-away in-memory database
conn.execute("PRAGMA foreign_keys = ON")

def q(sql):
    """Run a query, print formatted SQL, and display the result set as a table."""
    clean_sql = textwrap.dedent(sql).strip()

    # 1. Print formatted SQL query
    print("\n" + "=" * 60)
    print("QUERY:")
    print(textwrap.indent(clean_sql, "  "))
    print("-" * 60)

    cur = conn.execute(sql)
    rows = cur.fetchall()

    if cur.description:  # SELECT / statement returning rows
        headers = [d[0] for d in cur.description]
        str_rows = [[("" if v is None else str(v)) for v in r] for r in rows]

        # Calculate dynamic column widths for clean alignment
        widths = [
            max(len(h), *(len(r[i]) for r in str_rows), 1)
            for i, h in enumerate(headers)
        ] if str_rows else [len(h) for h in headers]

        header_line = " | ".join(h.ljust(widths[i]) for i, h in enumerate(headers))
        sep_line = "-+-".join("-" * widths[i] for i in range(len(headers)))

        print(header_line)
        print(sep_line)
        for r in str_rows:
            print(" | ".join(val.ljust(widths[i]) for i, val in enumerate(r)))
        if not str_rows:
            print("(0 rows returned)")
    else:  # DDL / DML (CREATE, INSERT, UPDATE, DELETE)
        affected = cur.rowcount
        print(f"Done (rows affected: {affected})" if affected != -1 else "Done.")

    print("=" * 60)

print('Connection ready:', conn)

dogs.to_sql('dogs', conn, index=False, if_exists='replace')
q("SELECT * FROM dogs")

## 🧪 SQL against CSV data — LIMIT and column selection

In [ ]:
q("SELECT name, breed FROM dogs LIMIT 2")
q("SELECT name, weight_kg FROM dogs WHERE weight_kg > 25 ORDER BY weight_kg")

## 🧪 Identifier quoting — the course backtick story

In [ ]:
# A column whose name is a reserved word needs quoting:
q('SELECT name AS "breed name" FROM dogs LIMIT 1')
try:
    conn.execute('SELECT name AS `my col` FROM dogs LIMIT 1').fetchall()   # backticks!
    print('SQLite accepts MySQL-style BACKTICK quoting too')
except sqlite3.OperationalError as e:
    print('backtick unsupported:', e)

## 💡 Tips & tricks

- Pipeline to remember: `pd.read_csv → df.to_sql → pd.read_sql_query` (or plain
  sqlite3 for one-off queries).
- If your CSV is huge: `chunksize=...` in read_csv + chunk-wise to_sql.
- Backtick habit from the course labs (`` `my-col` ``) is **MySQL/SQLite only** —
  PostgreSQL/Oracle use `"col"`.

## 🎯 Your turn

1. Make the CSV bigger (add 2 dogs), reload it and re-query in one go.
2. Run the whole round trip for a `cats.csv` you invent.

In [ ]:
# 💡 Solution
df2 = pd.concat([dogs, pd.DataFrame({'name':['Nala','Zeus'], 'breed':['Beagle','Corgi'], 'age':[4,1], 'weight_kg':[12.0,10.5]})], ignore_index=True)
d = tempfile.mkdtemp(); p = os.path.join(d, 'dogs2.csv')
df2.to_csv(p, index=False)
conn.execute('DROP TABLE IF EXISTS dogs')
pd.read_csv(p).to_sql('dogs', conn, index=False)
q("SELECT * FROM dogs")
df2.to_csv(os.path.join(tempfile.mkdtemp(), 'cats.csv'), index=False)
print('round trip for cats.csv done the same way — try it yourself with renamed columns.')